# Groundwater spatial sampling and validation diagnostics

This notebook uses the existing cleaned nitrate data to explore monitoring locations, nitrate differences between nearby wells, temporal coverage, and possible train/test leakage. It is exploratory: it does not train a model, select a validation design, change values, merge wells, remove rows, or regenerate buffers.

Distinct well IDs stay separate, including IDs at the same coordinates. The source observations and years remain intact.

## 1. Load and check the existing inputs

The checks confirm the saved nitrate table and 500 m buffer layer have the expected counts, years, and projected CRS. Distances and grid sizes need a CRS measured in meters, so the points use the buffer layer's CRS. The notebook reads existing buffers and does not recreate them.

In [1]:
from pathlib import Path

# Run this notebook from the repository root.
GROUNDWATER_DIR = Path('notebooks/groundwater')

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from IPython.display import display

DATA_PATH = GROUNDWATER_DIR / "groundwater_nitrate_clean.csv"
BUFFER_PATH = GROUNDWATER_DIR / "groundwater_well_buffers_500m.gpkg"
BUFFER_CSV_PATH = GROUNDWATER_DIR / "groundwater_well_buffers_500m.csv"

observations = pd.read_csv(
    DATA_PATH,
    dtype={"gm_well_id": "string", "src_samp_collection_time": "string"},
    parse_dates=["gm_samp_collection_date"],
)
assert len(observations) == 10_665, f"Unexpected observation count: {len(observations):,}"
assert observations["gm_well_id"].nunique() == 3_245
assert observations["gm_samp_collection_date"].notna().all()
assert observations[["gm_latitude", "gm_longitude"]].notna().all().all()

available_years = sorted(observations["gm_samp_collection_date"].dt.year.unique())
print(f"Observations: {len(observations):,}")
print(f"Unique well IDs: {observations['gm_well_id'].nunique():,}")
print(f"Years represented: {available_years[0]}?{available_years[-1]} ({len(available_years)} years)")
print(f"Source rows retained: {len(observations):,}")

buffers = gpd.read_file(BUFFER_PATH)
assert buffers.crs is not None, "The existing buffer layer has no CRS metadata."
assert len(buffers) == 3_245 and buffers["gm_well_id"].nunique() == 3_245
assert buffers.crs.to_epsg() == 3310, f"Unexpected buffer CRS: {buffers.crs}"
buffer_index = pd.read_csv(BUFFER_CSV_PATH, usecols=["gm_well_id", "buffer_m"])
assert len(buffer_index) == 3_245 and buffer_index["gm_well_id"].nunique() == 3_245
assert set(buffer_index["buffer_m"].dropna().unique()) == {500}
METRIC_CRS = buffers.crs
print(f"Existing buffers: {len(buffers):,}; buffer distance: 500 m; CRS: {METRIC_CRS}")

Observations: 10,665
Unique well IDs: 3,245
Years represented: 2010?2025 (16 years)
Source rows retained: 10,665
Existing buffers: 3,245; buffer distance: 500 m; CRS: EPSG:3310


## 2. Map monitoring locations and nearby wells

Each well ID has one recorded coordinate in this dataset. The notebook projects one point per ID into the buffer layer CRS (EPSG:3310), then summarizes nearest-neighbor distances, nearby IDs, and shared coordinates. The maps show sampling locations, not nitrate conditions. Shared coordinates do not prove that IDs are duplicates.

In [2]:
coordinate_rows = observations[["gm_well_id", "gm_longitude", "gm_latitude"]].drop_duplicates()
coordinate_counts = coordinate_rows.groupby("gm_well_id").size()
assert coordinate_counts.max() == 1, "At least one ID has conflicting coordinates; resolve before point-level summaries."

well_sites = gpd.GeoDataFrame(
    coordinate_rows.copy(),
    geometry=gpd.points_from_xy(coordinate_rows["gm_longitude"], coordinate_rows["gm_latitude"]),
    crs="EPSG:4326",
).to_crs(METRIC_CRS)
well_sites = well_sites.sort_values("gm_well_id").reset_index(drop=True)
well_xy = np.column_stack([well_sites.geometry.x, well_sites.geometry.y])
well_ids = well_sites["gm_well_id"].to_numpy()
well_tree = cKDTree(well_xy)
nearest_distances, nearest_indices = well_tree.query(well_xy, k=2)
nearest_other_m = nearest_distances[:, 1]
neighbor_counts = {}
for radius_m in [20, 30, 100, 500, 1_000]:
    neighbor_counts[f"neighbors_within_{radius_m}m"] = [
        len(found) - 1 for found in well_tree.query_ball_point(well_xy, r=radius_m)
    ]
neighbor_summary = pd.DataFrame({"gm_well_id": well_ids, "nearest_other_well_m": nearest_other_m, **neighbor_counts})

display(neighbor_summary[["nearest_other_well_m", *neighbor_counts]].describe(percentiles=[.1,.25,.5,.75,.9]).round(2))

shared_coordinates = coordinate_rows.groupby(["gm_longitude", "gm_latitude"])["gm_well_id"].agg(list)
shared_coordinates = shared_coordinates[shared_coordinates.map(len) > 1]
print(f"Coordinate pairs shared by multiple IDs: {len(shared_coordinates):,}")
print(f"IDs at shared coordinates: {sum(map(len, shared_coordinates)):,}")
display(shared_coordinates.rename("well_ids").reset_index().head(10))


fig = make_subplots(rows=1, cols=2, subplot_titles=("Unique well IDs in EPSG:3310", "Monitoring-location density"))
fig.add_trace(go.Scattergl(x=well_xy[:, 0], y=well_xy[:, 1], mode="markers", text=well_ids,
    marker={"size": 5, "color": neighbor_summary["neighbors_within_500m"], "colorscale": "Viridis",
            "colorbar": {"title": "Other IDs within 500 m"}}, name="Well ID"), row=1, col=1)
density, x_edges, y_edges = np.histogram2d(well_xy[:, 0], well_xy[:, 1], bins=35)
fig.add_trace(go.Heatmap(z=density.T, x=(x_edges[:-1]+x_edges[1:])/2,
    y=(y_edges[:-1]+y_edges[1:])/2, colorscale="Viridis", colorbar={"title": "Well IDs per grid cell"},
    name="Location count"), row=1, col=2)
fig.update_xaxes(title_text="Easting (m)", row=1, col=1); fig.update_yaxes(title_text="Northing (m)", scaleanchor="x", row=1, col=1)
fig.update_xaxes(title_text="Easting (m)", row=1, col=2); fig.update_yaxes(title_text="Northing (m)", scaleanchor="x2", row=1, col=2)
fig.update_layout(height=520, title="Monitoring locations and local sampling density")
display(fig)

display(go.Figure(go.Histogram(x=nearest_other_m, nbinsx=35, marker_color="#176b87")).update_layout(
    title="Nearest other well-ID distance", xaxis_title="Distance (m)", yaxis_title="Number of well IDs"))

,nearest_other_well_m,neighbors_within_20m,neighbors_within_30m,neighbors_within_100m,neighbors_within_500m,neighbors_within_1000m
count,3245.00,3245.00,3245.00,3245.00,3245.00,3245.00
mean,208.96,0.65,0.76,1.12,4.12,11.82
std,618.59,0.86,0.93,1.30,3.91,9.60
min,0.00,0.00,0.00,0.00,0.00,0.00
10%,2.83,0.00,0.00,0.00,0.00,2.00
25%,6.96,0.00,0.00,0.00,1.00,5.00
50%,27.94,0.00,1.00,1.00,3.00,9.00
75%,265.71,1.00,1.00,2.00,6.00,16.00
90%,525.56,2.00,2.00,3.00,9.00,23.00
max,20663.52,5.00,6.00,7.00,23.00,60.00


Coordinate pairs shared by multiple IDs: 50
IDs at shared coordinates: 104


,gm_longitude,gm_latitude,well_ids
0,-121.734450,36.686000,['AGC100000001-CCGC_0070' 'AGL020004618-CCGC_0...
1,-121.722960,36.678370,['AGC100000001-CCGC_0548' 'AGL020004641-CCGC_0...
2,-121.699750,36.676600,['AGC100000001-CCGC_0031' 'AGL020000974-CCGC_0...
3,-121.694720,36.671620,['AGC100000001-CCGC_0547' 'AGL020015807-CCGC_0...
4,-121.692020,36.647430,['AGC100000001-CCGC_0546' 'AGL020004640-CCGC_0...
5,-121.682188,36.654479,['AGL020028101-LANINI DUAL' 'AGL020035392-BARD...
6,-121.646840,36.642760,['AGC100000001-CCGC_0550' 'AGL020027408-CCGC_0...
7,-121.627550,36.627350,['AGC100000001-CCGC_0010' 'AGL020004048-CCGC_0...
8,-121.617332,36.590550,['AGL020023022-TARP_WEST_DOM' 'AGL020023022-TA...
9,-121.609470,36.603610,['AGC100000001-CCGC_0135' 'AGL020003706-CCGC_0...


### What the locations show

The nearest other ID is 27.94 m away at the median and 525.56 m at the 90th percentile. Fifty coordinate pairs are shared by 104 IDs. A well has a median of 9 other IDs within 1,000 m. Shared-coordinate IDs remain separate; their physical identity is unresolved.

## 3. Compare nitrate at nearby wells in similar periods

For same-day comparisons, repeated results for one well and date are averaged only in a temporary table. For same-year comparisons, each well's annual mean is used. Original rows stay unchanged. The comparisons include different IDs up to 1,000 m apart and are grouped by distance band. Pair-periods reuse wells, so they are not independent observations; proximity does not establish duplicate wells.

In [3]:
DISTANCE_EDGES = [0, 20, 30, 100, 500, 1_000.000001]
DISTANCE_LABELS = ["0?20 m", "20?30 m", "30?100 m", "100?500 m", "500?1,000 m"]

def nearby_period_pairs(period_data, period_name):
    """Summarize within-well period values, then compare distinct nearby IDs."""
    pair_rows = []
    for period, group in period_data.groupby("period", sort=False):
        summaries = group.groupby("gm_well_id", as_index=False).agg(
            nitrate_mg_L=("nitrate_mg_L", "mean"),
            source_records=("nitrate_mg_L", "size"),
            gm_longitude=("gm_longitude", "first"),
            gm_latitude=("gm_latitude", "first"),
        )
        if len(summaries) < 2:
            continue
        points = gpd.GeoDataFrame(
            summaries,
            geometry=gpd.points_from_xy(summaries.gm_longitude, summaries.gm_latitude),
            crs="EPSG:4326",
        ).to_crs(METRIC_CRS)
        xy = np.column_stack([points.geometry.x, points.geometry.y])
        pairs = cKDTree(xy).query_pairs(r=1_000, output_type="ndarray")
        if len(pairs) == 0:
            continue
        for left, right in pairs:
            a, b = summaries.iloc[left], summaries.iloc[right]
            pair_rows.append({
                "period": period,
                "well_id_a": a.gm_well_id,
                "well_id_b": b.gm_well_id,
                "distance_m": float(np.linalg.norm(xy[left] - xy[right])),
                "nitrate_a": float(a.nitrate_mg_L),
                "nitrate_b": float(b.nitrate_mg_L),
                "absolute_difference_mg_L": float(abs(a.nitrate_mg_L - b.nitrate_mg_L)),
                "records_a_summarized": int(a.source_records),
                "records_b_summarized": int(b.source_records),
            })
    pairs = pd.DataFrame(pair_rows)
    if pairs.empty:
        return pairs
    pairs["distance_band"] = pd.cut(
        pairs["distance_m"], bins=DISTANCE_EDGES, labels=DISTANCE_LABELS,
        right=False, include_lowest=True,
    )
    pairs["well_pair"] = pairs.apply(lambda row: "|".join(sorted([row.well_id_a, row.well_id_b])), axis=1)
    pairs["comparison_type"] = period_name
    return pairs

same_day_data = observations.assign(period=observations["gm_samp_collection_date"].dt.strftime("%Y-%m-%d"))
same_year_data = observations.assign(period=observations["gm_samp_collection_date"].dt.year)
same_day_pairs = nearby_period_pairs(same_day_data, "Same sampling date")
same_year_pairs = nearby_period_pairs(same_year_data, "Same calendar year; annual well means")

def summarize_pairs(pairs):
    if pairs.empty:
        return pd.DataFrame(columns=["distance_band", "comparisons", "unique_well_pairs", "median_abs_difference_mg_L", "p90_abs_difference_mg_L", "max_abs_difference_mg_L"])
    return pairs.groupby("distance_band", observed=True).agg(
        comparisons=("absolute_difference_mg_L", "size"),
        unique_well_pairs=("well_pair", "nunique"),
        median_abs_difference_mg_L=("absolute_difference_mg_L", "median"),
        p90_abs_difference_mg_L=("absolute_difference_mg_L", lambda x: x.quantile(.90)),
        max_abs_difference_mg_L=("absolute_difference_mg_L", "max"),
    ).reset_index()

daily_summary = summarize_pairs(same_day_pairs)
yearly_summary = summarize_pairs(same_year_pairs)
print(f"Same-date well-pair comparisons: {len(same_day_pairs):,}; same-year well-pair comparisons: {len(same_year_pairs):,}")
print("Same-date comparisons")
display(daily_summary.round(2))
print("Same-year annual-mean comparisons")
display(yearly_summary.round(2))

for label, pairs in [("Same date", same_day_pairs), ("Same year", same_year_pairs)]:
    close = pairs[pairs["distance_m"] < 20].sort_values("absolute_difference_mg_L", ascending=False)
    print(f"{label}: comparisons under 20 m = {len(close):,}")
    if not close.empty:
        display(close[["period", "well_id_a", "well_id_b", "distance_m", "nitrate_a", "nitrate_b", "absolute_difference_mg_L"]].head(10).round(3))
        print(f"Largest observed under-20-m difference ({label}): {close.iloc[0].absolute_difference_mg_L:.3f} mg/L")
    else:
        print("No under-20-m comparison was available in this period definition.")


fig = make_subplots(rows=1, cols=2, subplot_titles=("Same sampling date", "Same-year annual well means"), shared_yaxes=True)
for col, pairs in [(1, same_day_pairs), (2, same_year_pairs)]:
    for band in DISTANCE_LABELS:
        values = pairs.loc[pairs.distance_band == band, "absolute_difference_mg_L"]
        if len(values):
            fig.add_trace(go.Box(y=values, name=band, boxpoints=False, showlegend=False), row=1, col=col)
fig.update_layout(height=480, title="Observed nitrate differences between nearby wells", xaxis_title="Distance between IDs", yaxis_title="Absolute difference (mg/L)")
display(fig)

Same-date well-pair comparisons: 6,402; same-year well-pair comparisons: 19,231
Same-date comparisons


,distance_band,comparisons,unique_well_pairs,median_abs_difference_mg_L,p90_abs_difference_mg_L,max_abs_difference_mg_L
0,0?20 m,176,83,1.56,24.44,86.60
1,20?30 m,64,20,1.40,18.17,45.90
2,30?100 m,347,139,2.90,28.98,89.40
3,100?500 m,2276,902,3.80,28.48,226.20
4,"500?1,000 m",3539,1532,5.80,38.02,229.55


Same-year annual-mean comparisons


,distance_band,comparisons,unique_well_pairs,median_abs_difference_mg_L,p90_abs_difference_mg_L,max_abs_difference_mg_L
0,0?20 m,260,157,1.43,18.71,89.20
1,20?30 m,91,34,1.98,18.10,50.10
2,30?100 m,481,215,3.60,38.50,89.40
3,100?500 m,5185,2338,3.80,34.38,228.30
4,"500?1,000 m",13214,6337,5.30,39.40,229.55


Same date: comparisons under 20 m = 176


,period,well_id_a,well_id_b,distance_m,nitrate_a,nitrate_b,absolute_difference_mg_L
3620,2017-12-13,AGL020010825-CYPRESS OHB,AGL020010826-DOMESTIC,10.466,92.100,5.500,86.600
6152,2016-07-13,AGL020010825-CYPRESS OHB,AGL020010826-DOMESTIC,10.466,84.300,5.000,79.300
6162,2017-09-20,AGL020011783-WELL 4,AGL020011783-WELL DOM,8.110,92.213,15.047,77.166
3872,2017-06-23,AGL020011783-WELL 4,AGL020011783-WELL DOM,8.110,14.500,90.400,75.900
4398,2022-05-05,AGL020038542-SILA WELL1,AGL020038549-LAUR WELL1,0.000,56.900,2.500,54.400
4739,2025-04-10,AGL020038542-SILA WELL1,AGL020038549-LAUR WELL1,0.000,55.200,2.800,52.400
4818,2013-05-28,AGL020007884-AG_WELL_EN,AGL020018262-AG_WELL_YG,0.027,0.660,50.934,50.274
1917,2017-12-11,AGL020004084-NEW WELL,AGL020004084-WELL 2,5.707,58.100,13.800,44.300
6380,2020-07-21,AGL020030156-GV15 DOM,AGL020030156-GV15 IRR,2.397,51.000,7.600,43.400
3882,2017-06-23,AGL020004084-NEW WELL,AGL020004084-WELL 2,5.707,59.200,17.200,42.000


Largest observed under-20-m difference (Same date): 86.600 mg/L
Same year: comparisons under 20 m = 260


,period,well_id_a,well_id_b,distance_m,nitrate_a,nitrate_b,absolute_difference_mg_L
18453,2018,AGL020010825-CYPRESS OHB,AGL020010826-DOMESTIC,10.466,93.800,4.600,89.200
4848,2017,AGL020010825-CYPRESS OHB,AGL020010826-DOMESTIC,10.466,92.100,5.500,86.600
1712,2016,AGL020010825-CYPRESS OHB,AGL020010826-DOMESTIC,10.466,84.300,5.000,79.300
7860,2022,AGL020038542-SILA WELL1,AGL020038549-LAUR WELL1,0.000,56.900,2.500,54.400
16863,2025,AGL020038542-SILA WELL1,AGL020038549-LAUR WELL1,0.000,55.200,2.800,52.400
2099,2012,AGL020007884-AG_WELL_EN,AGL020018262-AG_WELL_YG,0.027,0.447,51.200,50.753
894,2013,AGL020007884-AG_WELL_EN,AGL020018262-AG_WELL_YG,0.027,0.660,50.934,50.274
18031,2020,AGL020030156-GV15 DOM,AGL020030156-GV15 IRR,2.397,51.000,7.600,43.400
4672,2017,AGL020004084-NEW WELL,AGL020004084-WELL 2,5.707,58.650,15.500,43.150
8802,2022,AGL020004185-HOME_IRR_1,AGL020004185-HOME_IRR_2,14.202,48.700,14.700,34.000


Largest observed under-20-m difference (Same year): 89.200 mg/L


### What the comparisons show

There are 6,402 same-date pair-period comparisons and 19,231 same-year comparisons. For different IDs less than 20 m apart, the largest absolute difference is 86.6 mg/L on the same date and 89.2 mg/L for annual means. Median differences are 1.56 and 1.43 mg/L; 90th percentiles are 24.44 and 18.71 mg/L. Many are small, but some are large. These comparisons are descriptive, not independent replicates.

## 4. Compare changes within and between nearby wells

The notebook summarizes annual means by well ID, calculates each well's range across those means, and compares it with same-year differences between nearby IDs. It also counts sampling dates and years per well. Ranges depend on when and how often a well was sampled, and these summaries use different samples; they are descriptive context, not a formal variance decomposition.

In [4]:
annual_well = observations.assign(year=observations["gm_samp_collection_date"].dt.year).groupby(
    ["gm_well_id", "year"], as_index=False
).agg(
    annual_mean_mg_L=("nitrate_mg_L", "mean"),
    observations=("nitrate_mg_L", "size"),
    sample_dates=("gm_samp_collection_date", "nunique"),
)
within_well = annual_well.groupby("gm_well_id").agg(
    years_observed=("year", "nunique"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    range_of_annual_means_mg_L=("annual_mean_mg_L", lambda x: x.max() - x.min()),
).reset_index()
coverage = observations.groupby("gm_well_id").agg(
    observations=("nitrate_mg_L", "size"),
    sample_dates=("gm_samp_collection_date", "nunique"),
    first_date=("gm_samp_collection_date", "min"),
    last_date=("gm_samp_collection_date", "max"),
).merge(within_well, on="gm_well_id")
print("Well temporal coverage")
display(coverage[["observations", "sample_dates", "years_observed", "range_of_annual_means_mg_L"]].describe(percentiles=[.25,.5,.75,.9]).round(2))

within_summary = within_well["range_of_annual_means_mg_L"].dropna().describe(percentiles=[.5,.9]).rename("Within-well range across annual means")
near_summary = same_year_pairs["absolute_difference_mg_L"].describe(percentiles=[.5,.9]).rename("Nearby different-well same-year absolute difference")
display(pd.concat([within_summary, near_summary], axis=1).round(2))


fig = make_subplots(rows=1, cols=2, subplot_titles=("Years with measurements per well ID", "Within-well range across annual means"))
fig.add_trace(go.Histogram(x=within_well["years_observed"], xbins={"size": 1}, marker_color="#176b87", showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=within_well["range_of_annual_means_mg_L"], nbinsx=35, marker_color="#d07b32", showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Distinct years", row=1, col=1); fig.update_yaxes(title_text="Well IDs", row=1, col=1)
fig.update_xaxes(title_text="Range (mg/L)", row=1, col=2); fig.update_yaxes(title_text="Well IDs", row=1, col=2)
fig.update_layout(height=430, title="Temporal coverage and within-well variability")
display(fig)

Well temporal coverage


,observations,sample_dates,years_observed,range_of_annual_means_mg_L
count,3245.00,3245.00,3245.00,3245.00
mean,3.29,3.28,2.89,4.53
std,2.71,2.71,1.97,10.19
min,1.00,1.00,1.00,0.00
25%,1.00,1.00,1.00,0.00
50%,3.00,3.00,2.00,0.60
75%,4.00,4.00,4.00,3.90
90%,6.00,6.00,6.00,13.96
max,30.00,30.00,12.00,154.90


,Within-well range across annual means,Nearby different-well same-year absolute difference
count,3245.00,19231.00
mean,4.53,13.66
std,10.19,22.23
min,0.00,0.00
50%,0.60,4.70
90%,13.96,37.60
max,154.90,229.55


### What the summaries show

Across 3,245 IDs, the median within-well range across annual means is 0.60 mg/L; the 90th percentile is 13.96 mg/L and the maximum is 154.90 mg/L. For nearby-well same-year comparisons up to 1 km, the median absolute difference is 4.70 mg/L and the 90th percentile is 37.60 mg/L. Different sampling patterns mean these values do not establish whether spatial or temporal variation dominates.

## 5. Illustrative train/test proximity checks

With a fixed seed, the notebook compares a random observation split, a split grouped by well ID, and a split holding out complete 5 km grid cells. It reports well overlap, row and well counts, nearest training-well distances, proximity thresholds, and year coverage. No model is fitted and no split is recommended. Random row splits can put the same repeatedly sampled well in both sets; grouping prevents that, while nearby different wells can still cross the split. Block edges can also remain close.

In [5]:
RANDOM_SEED = 42
TEST_FRACTION = 0.20
PROXIMITY_THRESHOLDS_M = [20, 30, 100, 500, 1_000]
row_index = np.arange(len(observations))
random_train_rows, random_test_rows = train_test_split(
    row_index, test_size=TEST_FRACTION, random_state=RANDOM_SEED, shuffle=True
)
group_splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=RANDOM_SEED)
group_train_rows, group_test_rows = next(group_splitter.split(
    row_index, groups=observations["gm_well_id"]
))

site_frame = well_sites[["gm_well_id", "geometry"]].copy().reset_index(drop=True)

def block_partition(site_data, block_size_m, offset_fraction=0.0, seed=RANDOM_SEED):
    xy = np.column_stack([site_data.geometry.x, site_data.geometry.y])
    offset = block_size_m * offset_fraction
    block_x = np.floor((xy[:, 0] - offset) / block_size_m).astype(int)
    block_y = np.floor((xy[:, 1] - offset) / block_size_m).astype(int)
    block_id = pd.Series([f"{x}:{y}" for x, y in zip(block_x, block_y)], index=site_data.index)
    split = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_i, test_i = next(split.split(site_data, groups=block_id))
    return site_data.iloc[train_i].gm_well_id.tolist(), site_data.iloc[test_i].gm_well_id.tolist(), block_id

def diagnose_split(name, train_rows, test_rows, train_site_ids=None, test_site_ids=None, block_ids=None):
    train_rows = np.asarray(train_rows, dtype=int)
    test_rows = np.asarray(test_rows, dtype=int)
    if train_site_ids is None:
        train_site_ids = observations.iloc[train_rows].gm_well_id.unique().tolist()
        test_site_ids = observations.iloc[test_rows].gm_well_id.unique().tolist()
    train_set, test_set = set(train_site_ids), set(test_site_ids)
    train_points = site_frame[site_frame.gm_well_id.isin(train_set)].reset_index(drop=True)
    test_points = site_frame[site_frame.gm_well_id.isin(test_set)].reset_index(drop=True)
    xy_train = np.column_stack([train_points.geometry.x, train_points.geometry.y])
    xy_test = np.column_stack([test_points.geometry.x, test_points.geometry.y])
    tree = cKDTree(xy_train)
    k = min(2, len(train_points))
    dist, idx = tree.query(xy_test, k=k)
    if k == 1:
        dist = np.asarray(dist).reshape(-1, 1)
        idx = np.asarray(idx).reshape(-1, 1)
    nearest_different = []
    for row, test_id in enumerate(test_points.gm_well_id):
        candidates = [(float(dist[row, j]), train_points.gm_well_id.iloc[int(idx[row, j])]) for j in range(k)]
        different = [value for value, train_id in candidates if train_id != test_id]
        nearest_different.append(min(different) if different else np.nan)
    nearest_different = np.asarray(nearest_different, dtype=float)
    train_obs = observations.iloc[train_rows]
    test_obs = observations.iloc[test_rows]
    years_train = sorted(train_obs.gm_samp_collection_date.dt.year.unique())
    years_test = sorted(test_obs.gm_samp_collection_date.dt.year.unique())
    result = {
        "strategy": name,
        "train_observations": len(train_rows), "test_observations": len(test_rows),
        "train_wells": len(train_set), "test_wells": len(test_set),
        "well_ID_overlap": len(train_set & test_set),
        "min_distance_different_wells_m": float(np.nanmin(nearest_different)),
        "median_nearest_training_well_m": float(np.nanmedian(nearest_different)),
        "train_years": f"{years_train[0]}?{years_train[-1]} ({len(years_train)})",
        "test_years": f"{years_test[0]}?{years_test[-1]} ({len(years_test)})",
    }
    for threshold in PROXIMITY_THRESHOLDS_M:
        result[f"test_wells_with_train_within_{threshold}m_pct"] = 100 * np.mean(nearest_different <= threshold)
    if block_ids is not None:
        result["train_blocks"] = block_ids.loc[site_frame.gm_well_id.isin(train_set)].nunique()
        result["test_blocks"] = block_ids.loc[site_frame.gm_well_id.isin(test_set)].nunique()
    return result

random_metrics = diagnose_split("Random observations", random_train_rows, random_test_rows)
group_metrics = diagnose_split("Group by well ID", group_train_rows, group_test_rows)
block_train_ids, block_test_ids, block_ids_5km = block_partition(site_frame, 5_000)
block_train_rows = np.flatnonzero(observations.gm_well_id.isin(block_train_ids).to_numpy())
block_test_rows = np.flatnonzero(observations.gm_well_id.isin(block_test_ids).to_numpy())
block_metrics = diagnose_split("5 km spatial blocks (origin grid)", block_train_rows, block_test_rows, block_train_ids, block_test_ids, block_ids_5km)
split_diagnostics = pd.DataFrame([random_metrics, group_metrics, block_metrics])
display(split_diagnostics.round(2))


fig = go.Figure()
for threshold in PROXIMITY_THRESHOLDS_M:
    col = f"test_wells_with_train_within_{threshold}m_pct"
    fig.add_trace(go.Bar(x=split_diagnostics["strategy"], y=split_diagnostics[col], name=f"? {threshold} m"))
fig.update_layout(barmode="group", height=450, title="Test-well proximity to a different training well", yaxis_title="Test wells (%)", xaxis_title="Illustrative split")
display(fig)

,strategy,train_observations,test_observations,train_wells,test_wells,well_ID_overlap,min_distance_different_wells_m,median_nearest_training_well_m,train_years,test_years,test_wells_with_train_within_20m_pct,test_wells_with_train_within_30m_pct,test_wells_with_train_within_100m_pct,test_wells_with_train_within_500m_pct,test_wells_with_train_within_1000m_pct,train_blocks,test_blocks
0,Random observations,8532,2133,3045,1474,1274,0.00,57.98,2010?2025 (16),2012?2025 (14),38.67,43.96,55.22,88.87,97.56,NaN,NaN
1,Group by well ID,8601,2064,2596,649,0,0.00,68.50,2010?2025 (16),2012?2025 (14),38.52,43.30,54.08,84.44,96.61,NaN,NaN
2,5 km spatial blocks (origin grid),9045,1620,2769,476,0,44.54,1431.48,2011?2025 (15),2010?2025 (15),0.00,0.00,0.42,9.24,31.30,112.0,29.0


### What the split checks show

The random observation split puts 1,274 of 1,474 test IDs in both sets. Grouping by well ID removes ID overlap, but the median nearest different training well is 68.50 m. The illustrative 5 km block split also has no ID overlap; its median nearest training well is 1,431.48 m, while 31.30% of test wells remain within 1 km of another training well.

## 6. Compare block sizes and grid origins

The notebook compares 1, 5, and 10 km square grids from the origin and a half-cell offset. For each candidate, a fixed-seed 20% block holdout is drawn. The table reports occupied blocks, wells and observations in each set, balance, and cross-set proximity. These are candidates for discussion, not tuned choices or a final validation design.

In [6]:
block_results = []
block_assignments = {}
for size_m in [1_000, 5_000, 10_000]:
    for offset_fraction, label in [(0.0, "origin"), (0.5, "half-cell offset")]:
        train_ids, test_ids, ids_by_site = block_partition(
            site_frame, size_m, offset_fraction=offset_fraction, seed=RANDOM_SEED
        )
        train_rows = np.flatnonzero(observations.gm_well_id.isin(train_ids).to_numpy())
        test_rows = np.flatnonzero(observations.gm_well_id.isin(test_ids).to_numpy())
        metrics = diagnose_split(
            f"{size_m/1000:g} km, {label}", train_rows, test_rows,
            train_ids, test_ids, ids_by_site,
        )
        metrics["block_size_km"] = size_m / 1_000
        metrics["grid_origin"] = label
        metrics["occupied_blocks"] = ids_by_site.nunique()
        metrics["test_observation_share_pct"] = 100 * len(test_rows) / len(observations)
        block_results.append(metrics)
        block_assignments[(size_m, label)] = (train_ids, test_ids)
block_comparison = pd.DataFrame(block_results)
display(block_comparison[[
    "block_size_km", "grid_origin", "occupied_blocks", "train_blocks", "test_blocks",
    "train_wells", "test_wells", "train_observations", "test_observations",
    "test_observation_share_pct", "well_ID_overlap", "min_distance_different_wells_m",
    "median_nearest_training_well_m", *[f"test_wells_with_train_within_{x}m_pct" for x in [20,30,100,500,1000]],
]].round(2))


fig = go.Figure()
for origin, group in block_comparison.groupby("grid_origin"):
    fig.add_trace(go.Scatter(x=group.block_size_km, y=group.test_observation_share_pct, mode="lines+markers", name=origin))
fig.add_hline(y=TEST_FRACTION*100, line_dash="dash", line_color="gray", annotation_text="Target share (20%)")
fig.update_layout(height=420, title="Block size and test observation share", xaxis_title="Candidate square block width (km)", yaxis_title="Test observations (%)")
display(fig)


fig = go.Figure()
for origin, group in block_comparison.groupby("grid_origin"):
    fig.add_trace(go.Scatter(x=group.block_size_km, y=group.min_distance_different_wells_m, mode="lines+markers", name=origin))
fig.update_layout(height=420, title="Block size and closest cross-partition wells", xaxis_title="Candidate square block width (km)", yaxis_title="Minimum distance (m)")
display(fig)

,block_size_km,grid_origin,occupied_blocks,train_blocks,test_blocks,train_wells,test_wells,train_observations,test_observations,test_observation_share_pct,well_ID_overlap,min_distance_different_wells_m,median_nearest_training_well_m,test_wells_with_train_within_20m_pct,test_wells_with_train_within_30m_pct,test_wells_with_train_within_100m_pct,test_wells_with_train_within_500m_pct,test_wells_with_train_within_1000m_pct
0,1.0,origin,1027,821,206,2605,640,8520,2145,20.11,0,19.36,600.34,0.16,0.16,1.88,33.28,90.16
1,1.0,half-cell offset,1018,814,204,2626,619,8582,2083,19.53,0,11.36,598.43,0.48,0.81,2.58,34.89,90.15
2,5.0,origin,141,112,29,2769,476,9045,1620,15.19,0,44.54,1431.48,0.00,0.00,0.42,9.24,31.30
3,5.0,half-cell offset,143,114,29,2804,441,9174,1491,13.98,0,21.78,1213.97,0.00,0.23,0.68,9.98,39.23
4,10.0,origin,59,47,12,2517,728,8413,2252,21.12,0,119.03,2544.71,0.00,0.00,0.00,4.95,18.54
5,10.0,half-cell offset,61,48,13,2728,517,9193,1472,13.80,0,245.54,3705.28,0.00,0.00,0.00,0.97,7.35


### How block choices affect proximity

For 1 km blocks, 90.15 to 90.16% of test wells have another training well within 1 km, and the minimum cross-set distance is 11 to 19 m. For 5 km blocks, the share is 31.30 to 39.23%; for 10 km blocks, 7.35 to 18.54%. Grid shifts change both test share and proximity. The 5 km test share is 15.19% at the origin and 13.98% with the offset; for 10 km it is 21.12% and 13.80%. No candidate is selected; balance, intended use, and grid sensitivity still need team discussion.

## 7. Keep three tasks separate

Spatial declustering changes how locations represent a target population or surface. Spatial cross-validation changes how observations are assigned to training and test sets. Record deduplication removes records only when identifiers and provenance establish that they are duplicates. Nearby wells can be distinct and have different nitrate values, so distance alone does not justify weighting or deletion.

The analyses retain every nitrate row and year. They describe location density, observed proximity, concentration differences, and illustrative split behavior. They do not establish duplicates, causal spatial effects, an optimal block size, declustering weights, or expected model performance. Discuss the prediction target and deployment area before choosing an approach.

## 8. Reproducibility and data preservation

The nitrate table and existing buffer layer are read only. Annual summaries and split assignments stay in memory; no data, figures, models, or predictions are written. The random seed is 42.

In [7]:
assert len(observations) == 10_665
assert observations.gm_well_id.nunique() == 3_245
assert sorted(observations.gm_samp_collection_date.dt.year.unique()) == available_years
assert split_diagnostics.well_ID_overlap.iloc[1] == 0  # Group-by-ID split
print("All source observations and years remain represented in memory.")
print("No export or dataset-write operation is used in this notebook.")


All source observations and years remain represented in memory.
No export or dataset-write operation is used in this notebook.
